# Visualizing the images used to compute the metrics

This notebook is **standalone**: it `git clone`s/`git pull`s the `galaxy-morphometrics` repo, then displays 10 postage stamps across 4 columns:

1. **Raw data** — the stamp as loaded from the Hugging Face dataset.
2. **Data with mask applied** — the same stamp after the processing applied when `--mask-field` is used: bad pixels (red contour) are replaced by the local estimate (`galmorph.stats._fill_masked`, local median) used internally for the CAS/Gini-M20/MID indicators.
3. **Raw reconstruction** — the autoencoder's output (`Autoencoder.reconstruct`), noise-free.
4. **Reconstruction + noise** — the same reconstruction after `galmorph.data.add_noise` (white noise scaled by the `noise_map`).

Display: no normalization by default (`imshow` picks `vmin`/`vmax` per panel). The `SHARED_ROW_SCALE` option (config cell) lets you force a common `vmin`/`vmax` per row (raw min/max of the initial stamp, no stretch or ZScale) if you want to compare levels across columns.

Dependencies: neither `galsim` nor R is used here. `WandBGalaxyAutoencoder` needs `pshear` and Train-AE's requirements (see the README).

In [ ]:
# --- Config: adapt before running ---

REPO_URL = "https://github.com/CosmoStat/galaxy-morphometrics.git"
REPO_DIR = "galaxy-morphometrics"  # local folder (created next to the notebook)
GITHUB_TOKEN = None  # set a token if the repo is private, e.g. os.environ["GITHUB_TOKEN"]

# --- Hugging Face dataset ---
DATASET = "your-org/your-dataset"       # <-- replace
HF_CONFIG = None
SPLIT = "train"
IMAGE_FIELD = "image"                   # <-- adapt to the dataset's column name
MASK_FIELD = "mask"                     # mask column (1=valid/0=defective), or None to disable
PSF_FIELD = "psf_stamp"                 # PSF column (needed by WandBGalaxyAutoencoder), or None
NOISE_MAP_FIELD = "noise_map"           # per-pixel noise std column, or None
N_SAMPLES = 10                          # exactly the 10 images to display
STAMP_SIZE = 64
NOISE_SEED = 0
HF_TOKEN = None                         # or os.environ.get("HF_TOKEN")

# --- Autoencoder (set to None to only display columns 1 and 2) ---
AUTOENCODER_SPEC = "galmorph.autoencoder:WandBGalaxyAutoencoder"
ENCODER_PATH = "entity/project/run_id"  # WandB run_path
DECODER_PATH = "1400"                   # checkpoint epoch

# --- Display ---
# False: each panel gets its own vmin/vmax. True: each row uses the raw min/max of
# its original stamp, to compare levels across columns.
SHARED_ROW_SCALE = False

## 1. Fetching the repo (git clone / git pull)

In [ ]:
import os
import subprocess
import sys

repo_url = REPO_URL
if GITHUB_TOKEN:
    repo_url = REPO_URL.replace("https://", "https://%s@" % GITHUB_TOKEN)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, git pull...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

## 2. Imports

In [ ]:
import importlib

import numpy as np
import matplotlib.pyplot as plt

from galmorph.data import add_noise, load_hf_stamps
from galmorph.stats import _fill_masked


def build_autoencoder(spec, encoder_path, decoder_path):
    module_name, class_name = spec.split(":")
    cls = getattr(importlib.import_module(module_name), class_name)
    args = [a for a in (encoder_path, decoder_path) if a is not None]
    return cls(*args)

## 3. Loading the 10 stamps (+ mask / PSF / noise map if configured)

In [ ]:
loaded = load_hf_stamps(
    DATASET,
    split=SPLIT,
    image_field=IMAGE_FIELD,
    n_samples=N_SAMPLES,
    stamp_size=STAMP_SIZE,
    hf_config=HF_CONFIG,
    hf_token=HF_TOKEN,
    psf_field=PSF_FIELD,
    noise_map_field=NOISE_MAP_FIELD,
    mask_field=MASK_FIELD,
)

if PSF_FIELD or NOISE_MAP_FIELD or MASK_FIELD:
    real_images, extra = loaded
else:
    real_images, extra = loaded, {}

psf_images = extra.get("psf")
noise_map = extra.get("noise_map")
real_masks = extra.get("mask")  # convention: nonzero = bad pixel (already flipped by load_hf_stamps)

print("Loaded %d stamps of size %dx%d" % (len(real_images), STAMP_SIZE, STAMP_SIZE))

## 4. Reconstruction (raw + with noise)

In [ ]:
if AUTOENCODER_SPEC:
    ae = build_autoencoder(AUTOENCODER_SPEC, ENCODER_PATH, DECODER_PATH)
    recon_kwargs = {"psf": psf_images} if psf_images is not None else {}
    reconstruction = ae.reconstruct(real_images, **recon_kwargs)

    if noise_map is not None:
        reconstruction_noisy = add_noise(reconstruction, noise_map, seed=NOISE_SEED)
    else:
        print("NOISE_MAP_FIELD not configured -- column 4 is identical to column 3")
        reconstruction_noisy = reconstruction
else:
    print("AUTOENCODER_SPEC=None -- only columns 1 and 2 will be displayed")
    reconstruction = None
    reconstruction_noisy = None

## 5. "Real" data as modified by the mask parameter

Reproduces exactly the processing done by `galmorph.stats.morph_stats`: pixels flagged as bad are replaced by the local median (`_fill_masked`) before computing the indicators, instead of being left at their raw value.

In [ ]:
if real_masks is not None:
    masked_images = np.stack(
        [
            _fill_masked(real_images[i], real_masks[i] != 0) if (real_masks[i] != 0).any() else real_images[i]
            for i in range(len(real_images))
        ]
    )
else:
    print("MASK_FIELD not configured -- column 2 is identical to column 1")
    masked_images = real_images

## 6. 10-image x 4-column grid

In [ ]:
columns = [("Raw data", real_images), ("Data with mask", masked_images)]
if reconstruction is not None:
    columns.append(("Raw reconstruction", reconstruction))
    columns.append(("Reconstruction + noise", reconstruction_noisy))

n_rows = len(real_images)
n_cols = len(columns)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]
if n_cols == 1:
    axes = axes[:, np.newaxis]

for row in range(n_rows):
    if SHARED_ROW_SCALE:
        vmin, vmax = real_images[row].min(), real_images[row].max()
        imshow_kwargs = {"vmin": vmin, "vmax": vmax}
    else:
        imshow_kwargs = {}  # let imshow pick vmin/vmax per panel, no normalization

    for col, (title, stack) in enumerate(columns):
        ax = axes[row, col]
        ax.imshow(stack[row], origin="lower", cmap="viridis", **imshow_kwargs)
        ax.set_xticks([])
        ax.set_yticks([])
        if title == "Data with mask" and real_masks is not None:
            bad = real_masks[row] != 0
            if bad.any():
                ax.contour(bad, levels=[0.5], colors="red", linewidths=0.8)
        if row == 0:
            ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.show()